<a href="https://colab.research.google.com/github/shivanshi-09/IML_Midterm/blob/main/Ordinal_XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
import warnings
warnings.filterwarnings('ignore')
df = pd.read_csv("https://raw.githubusercontent.com/shivanshi-09/IML_Midterm/main/nhanes_diabetes_clean.csv")
print(f'Loaded: {df.shape[0]} rows, {df.shape[1]} columns')
print('Label distribution:')
print(df['diabetes_label'].value_counts().sort_index().rename({0:'Normal',1:'Pre-diab',2:'Diabetic'}))

Loaded: 6045 rows, 30 columns
Label distribution:
diabetes_label
Normal      3659
Pre-diab    1635
Diabetic     751
Name: count, dtype: int64


In [ ]:
TARGET_COL = 'diabetes_label'

FEATURE_COLS_T1 = ['RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'BMXBMI', 'BMXWAIST', 'BPXSY_mean', 'BPXDI_mean']
FEATURE_COLS_T2 = ['LBXTC', 'LBXTR', 'LBXSCR', 'LBXSUA', 'LBXSBU']
FEATURE_COLS_T3 = ['LBXSATSI', 'LBXSASSI', 'LBXSTP', 'LBXSAL', 'LBXSCA', 'LBXSPH', 'LBXSNASI', 'LBXSKSI', 'LBXSGB', 'LBXSC3SI']
ALL_FEATURE_COLS = FEATURE_COLS_T1 + FEATURE_COLS_T2 + FEATURE_COLS_T3

DEFAULT_CONFIG = {
    'learning_rate':    0.1,
    'max_depth':        6,
    'n_estimators':     300,
    'colsample_bytree': 0.75,
    'subsample':        0.8,
}

TEST_SIZE   = 0.2
RANDOM_SEED = 42

In [ ]:
def split_data(df, feature_cols=ALL_FEATURE_COLS):
    X = df[feature_cols].values
    y = df[TARGET_COL].values
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y)
    X_train, X_cal, y_train, y_cal = train_test_split(
        X_temp, y_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=y_temp)
    print(f'Train: {X_train.shape[0]} | Cal: {X_cal.shape[0]} | Test: {X_test.shape[0]}')
    return X_train, X_cal, X_test, y_train, y_cal, y_test

X_train, X_cal, X_test, y_train, y_cal, y_test = split_data(df)

Train: 3627 | Cal: 1209 | Test: 1209


In [ ]:
class OrdinalXGBoost:
    def __init__(self, n_classes=3, **xgb_kwargs):
        self.n_classes  = n_classes
        self.xgb_kwargs = xgb_kwargs
        self.models     = []

    def fit(self, X, y, sample_weight=None):
        self.models = []
        for k in range(self.n_classes - 1):
            y_bin = (y > k).astype(int)
            m = XGBClassifier(eval_metric='logloss', **self.xgb_kwargs)
            m.fit(X, y_bin, sample_weight=sample_weight)
            self.models.append(m)
        return self

    def predict_proba(self, X):
        cm = np.column_stack([m.predict_proba(X)[:, 1] for m in self.models])
        cm = np.minimum.accumulate(cm, axis=1)
        probs = np.zeros((X.shape[0], self.n_classes))
        probs[:, 0]  = 1.0 - cm[:, 0]
        for k in range(1, self.n_classes - 1):
            probs[:, k] = cm[:, k - 1] - cm[:, k]
        probs[:, -1] = cm[:, -1]
        probs = np.clip(probs, 1e-9, 1.0)
        probs = probs / probs.sum(axis=1, keepdims=True)
        return probs

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)


In [ ]:
sw_train = compute_sample_weight(class_weight='balanced', y=y_train)

std_model = XGBClassifier(**DEFAULT_CONFIG, objective='multi:softprob',
                          num_class=3, random_state=RANDOM_SEED)
std_model.fit(X_train, y_train, sample_weight=sw_train)

ord_model = OrdinalXGBoost(n_classes=3, **DEFAULT_CONFIG, random_state=RANDOM_SEED)
ord_model.fit(X_train, y_train, sample_weight=sw_train)

print('Both models trained on full feature set.')

Both models trained on full feature set.


In [ ]:
std_preds = std_model.predict(X_test)
ord_preds = ord_model.predict(X_test)

print('Standard XGBoost')
print(classification_report(y_test, std_preds, target_names=['Normal','Pre-diab','Diabetic'], digits=3))
print('Ordinal XGBoost')
print(classification_report(y_test, ord_preds, target_names=['Normal','Pre-diab','Diabetic'], digits=3))

print(f'Macro F1   |  Std: {f1_score(y_test, std_preds, average="macro"):.4f}   |  Ord: {f1_score(y_test, ord_preds, average="macro"):.4f}')
print(f'MAE (rank) |  Std: {np.abs(y_test - std_preds).mean():.4f}   |  Ord: {np.abs(y_test - ord_preds).mean():.4f}')
std_d2 = ((y_test == 0) & (std_preds == 2)).sum() + ((y_test == 2) & (std_preds == 0)).sum()
ord_d2 = ((y_test == 0) & (ord_preds == 2)).sum() + ((y_test == 2) & (ord_preds == 0)).sum()
print(f'\nDistance-2 errors (Normal<->Diabetic): Std={std_d2}, Ord={ord_d2}')

print('\n Confusion matrices (rows=true, cols=pred)')
print('Standard:')
print(confusion_matrix(y_test, std_preds))
print('Ordinal:')
print(confusion_matrix(y_test, ord_preds))


Standard XGBoost
              precision    recall  f1-score   support

      Normal      0.780     0.792     0.786       732
    Pre-diab      0.438     0.453     0.445       327
    Diabetic      0.449     0.380     0.412       150

    accuracy                          0.649      1209
   macro avg      0.555     0.542     0.548      1209
weighted avg      0.646     0.649     0.647      1209

Ordinal XGBoost
              precision    recall  f1-score   support

      Normal      0.825     0.755     0.789       732
    Pre-diab      0.447     0.596     0.511       327
    Diabetic      0.466     0.320     0.379       150

    accuracy                          0.658      1209
   macro avg      0.580     0.557     0.560      1209
weighted avg      0.679     0.658     0.663      1209

Macro F1   |  Std: 0.5475   |  Ord: 0.5598
MAE (rank) |  Std: 0.3962   |  Ord: 0.3747

Distance-2 errors (Normal<->Diabetic): Std=55, Ord=40

 Confusion matrices (rows=true, cols=pred)
Standard:
[[580 130 